In [0]:
MERGE INTO youtube_lakehouse.gold.video_history AS target
USING (
  SELECT 
    Title,
    publishedAt,
    snapshot_date,
    maxViews,
    DoDViews_Delta,
    DoDLikes_Delta,
    DoDComment_Delta
  FROM (
    SELECT 
      video_title AS Title,
      DATE(published_at) AS publishedAt,
      snapshot_date,
      cumulative_views AS maxViews,
      delta_views_24h AS DoDViews_Delta,
      delta_likes_24h AS DoDLikes_Delta,
      delta_comments_24h AS DoDComment_Delta,
      ROW_NUMBER() OVER (PARTITION BY video_title, DATE(published_at), snapshot_date ORDER BY cumulative_views DESC) AS row_num
    FROM youtube_lakehouse.silver.fact_video_daily_snapshots
    WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM youtube_lakehouse.silver.fact_video_daily_snapshots)
  ) ranked
  WHERE row_num = 1
) AS source
ON target.Title = source.Title 
   AND target.snapshot_date = source.snapshot_date
   AND target.publishedAt = source.publishedAt
WHEN MATCHED THEN
  UPDATE SET
    target.maxViews = source.maxViews,
    target.DoDViews_Delta = source.DoDViews_Delta,
    target.DoDLikes_Delta = source.DoDLikes_Delta,
    target.DoDComment_Delta = source.DoDComment_Delta
WHEN NOT MATCHED THEN
  INSERT (Title, publishedAt, snapshot_date, maxViews, DoDViews_Delta, DoDLikes_Delta, DoDComment_Delta)
  VALUES (source.Title, source.publishedAt, source.snapshot_date, source.maxViews, source.DoDViews_Delta, source.DoDLikes_Delta, source.DoDComment_Delta)